# Signature Detection - YOLO11 App in Google Colab
Run the cells **one by one, from top to bottom**.

Before you start: `Runtime > Change runtime type > T4 GPU`.

## Step 1: Install packages

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics fastapi "uvicorn[standard]" python-multipart

## Step 2: Upload the project zip
Click **Choose files** and select `yolo11-detection-app.zip`.

In [ ]:
from google.colab import files
import os, shutil, zipfile

uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content")

PROJECT = "/content/signature_detection_project"
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
os.rename("/content/yolo11-detection-app", PROJECT)
print(os.listdir(PROJECT))

## Step 3: Put `best.pt` in the weights folder
This cell looks for the model in this order:
1. the training run of this session, 2. your Google Drive, 3. asks you to upload it.

In [ ]:
import os, shutil

dst = f"{PROJECT}/weights/best.pt"
os.makedirs(f"{PROJECT}/weights", exist_ok=True)

candidates = [
    "/content/runs/signature/weights/best.pt",
    "/content/runs/detect/signature/weights/best.pt",
    "/content/drive/MyDrive/signature_detection_project/best.pt",
]
found = next((c for c in candidates if os.path.exists(c)), None)

if found:
    shutil.copy(found, dst)
    print("Model copied from:", found)
else:
    from google.colab import files
    print("Model not found. Please upload best.pt")
    up = files.upload()
    shutil.move(next(iter(up)), dst)

print("Model ready:", os.path.exists(dst), "|", round(os.path.getsize(dst)/1e6, 1), "MB")

### Optional: identity model (tells whose signature it is)
If you trained `people_cls.pt` with `colab_train_people.ipynb`, this cell adds it. Without it the app only detects signatures.

In [ ]:
people_dst = f"{PROJECT}/weights/people_cls.pt"
people_candidates = [
    f"{PROJECT}/runs/people/weights/best.pt",
    "/content/drive/MyDrive/signature_detection_project/people_cls.pt",
]
found_people = next((c for c in people_candidates if os.path.exists(c)), None)
if found_people:
    shutil.copy(found_people, people_dst)
    print("Identity model ready.")
else:
    print("No identity model found. The app will only detect signatures.")
    print("To add one: upload people_cls.pt to the weights folder (left Files panel).")

## Step 4: Start the server

In [ ]:
import subprocess, time, requests

os.chdir(PROJECT)
log = open("/content/server.log", "w")
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT, cwd=PROJECT,
)

for _ in range(60):
    try:
        print(requests.get("http://127.0.0.1:8000/health", timeout=2).json())
        break
    except Exception:
        time.sleep(2)
else:
    print("Server did not start. Log:\n")
    print(open("/content/server.log").read()[-3000:])

## Step 5: Open the app
You should see the web page below with a green badge **"Server online"**.
If the page is blank, run the next cell instead (opens in a new tab).

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=950)

In [ ]:
# Alternative: open the app in a new browser tab
from google.colab import output
output.serve_kernel_port_as_window(8000)

## Optional: public link (to show friends or teachers)
Anyone with the link can use your app **while this notebook is running**. The link changes every time.

In [ ]:
import re, time, subprocess, os

if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

tlog = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=tlog, stderr=subprocess.STDOUT,
)

url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break
print("Public link:", url or "not ready, run this cell again")

## Save the model to Google Drive (recommended)
Colab deletes all files when the session ends. Run this once so `best.pt` is never lost.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

folder = "/content/drive/MyDrive/signature_detection_project"
os.makedirs(folder, exist_ok=True)
shutil.copy(f"{PROJECT}/weights/best.pt", f"{folder}/best.pt")
print("Saved to Drive:", folder)

## Stop the server (when you are done)

In [ ]:
server.terminate()
try:
    tunnel.terminate()
except NameError:
    pass
print("Stopped.")